# Terwilliger algebra T(G)

Computes, for the group association scheme of a finite group `G`:
Bose–Mesner algebra 𝔄(G), its dual 𝔄*(G), the intersection numbers,
the full Terwilliger algebra T(G) = ⟨𝔄, 𝔄*⟩, its center Z(T(G)),
the primitive central idempotents ε_i ("the epsilons"), and the
Wedderburn degrees d_i (so T(G) ≅ ⊕ M_{d_i}(ℂ)).

In [67]:
G = SymmetricGroup(5)
G

Symmetric group of order 5! as a permutation group

In [68]:
classes = [ccl.list() for ccl in G.conjugacy_classes()]
elements = []
for ccl in classes:
    elements += ccl

nc = len(classes)         
N  = len(elements)

class_of = {}
for i, ccl in enumerate(classes):
    for g in ccl:
        class_of[g] = i

print(f"{G} has order {N} and {nc} conjugacy classes:")
for i, ccl in enumerate(classes):
    print(f"  C{i}: size {len(ccl)}, representative {ccl[0]}")

Symmetric group of order 5! as a permutation group has order 120 and 7 conjugacy classes:
  C0: size 1, representative ()
  C1: size 10, representative (2,4)
  C2: size 15, representative (2,3)(4,5)
  C3: size 20, representative (2,4,5)
  C4: size 20, representative (1,3)(2,4,5)
  C5: size 30, representative (2,3,4,5)
  C6: size 24, representative (1,2,3,4,5)


In [69]:
Amats = [matrix(QQ, N, N) for _ in range(nc)]
for x in range(N):
    gx = elements[x]
    for y in range(N):
        c = class_of[gx * elements[y].inverse()]
        Amats[c][x, y] = 1

Estar = []
for i in range(nc):
    diag = [1 if class_of[g] == i else 0 for g in elements]
    Estar.append(diagonal_matrix(QQ, diag))

print("sum A_i == all-ones matrix:", sum(Amats) == matrix(QQ, N, N, lambda i,j: 1))
print("sum E_i* == identity:", sum(Estar) == identity_matrix(QQ, N))

sum A_i == all-ones matrix: True
sum E_i* == identity: True


In [70]:
def rep_positions(Amats):
    """One representative (row,col) position per class, read off A_i."""
    pos = []
    for A in Amats:
        found = None
        for r in range(A.nrows()):
            for c in range(A.ncols()):
                if A[r, c] == 1:
                    found = (r, c); break
            if found: break
        pos.append(found)
    return pos

def intersection_numbers(Amats):
    nc = len(Amats)
    idx = rep_positions(Amats)
    P = []
    for i in range(nc):
        rows = []
        for j in range(nc):
            Aij = Amats[i] * Amats[j]
            rows.append([Aij[r, c] for (r, c) in idx])
        P.append(matrix(rows))
    return P

def regular_rep_matrices(P):
    nc = len(P)
    B = []
    for i in range(nc):
        Bi = matrix(nc, nc)
        for j in range(nc):
            for k in range(nc):
                Bi[j, k] = P[k][i, j]
        B.append(Bi)
    return B

Pmats = intersection_numbers(Amats)
Bmats = regular_rep_matrices(Pmats)

dim_To = sum(1 for Bi in Bmats for x in Bi.list() if x != 0)
print("dim T_o(G) =", dim_To)

dim T_o(G) = 124


In [71]:
def simultaneous_diagonalizer(mats):
    P = mats[0].eigenmatrix_right()[1]
    for M in mats[1:]:
        Mtmp = P.inverse() * M * P
        Pt = Mtmp.eigenmatrix_right()[1]
        P = P * Pt
    return P

def eigenvalue_matrix(P, mats):
    Pinv = P.inverse()
    n = len(mats)
    M = matrix(Pinv.base_ring(), n, n)
    for i, X in enumerate(mats):
        M[:, i] = matrix((Pinv * X * P).diagonal()).transpose()[:, 0]
    return M

def idempotents_from(M, mats):
    Minv = M.inverse()
    n = len(mats)
    return [sum(Minv[i, j] * mats[i] for i in range(n)) for j in range(n)]

In [72]:
Pdiag_BM = simultaneous_diagonalizer(Bmats)
Meig_BM  = eigenvalue_matrix(Pdiag_BM, Bmats)
E_idem   = idempotents_from(Meig_BM, Amats)

print("all idempotent:", all((e*e - e).is_zero() for e in E_idem))
print("sum == identity:", sum(E_idem) == identity_matrix(QQ, N))

all idempotent: True
sum == identity: True


In [73]:
def terwilliger_spanning_set(Amats, Estar):
    nc = len(Amats)
    EA  = [[Estar[i] * Amats[j] for j in range(nc)] for i in range(nc)]
    EAE = [[[EA[i][j] * Estar[k] for k in range(nc)] for j in range(nc)] for i in range(nc)]
    AE  = [[Amats[l] * Estar[m] for m in range(nc)] for l in range(nc)]

    S = []
    for i in range(nc):
        for j in range(nc):
            for k in range(nc):
                left = EAE[i][j][k]
                if left.is_zero():
                    continue       
                for l in range(nc):
                    for m in range(nc):
                        Mprod = left * AE[l][m]
                        if not Mprod.is_zero():
                            S.append(Mprod)
    return S

spanning_set = terwilliger_spanning_set(Amats, Estar)
print("nonzero candidates:", len(spanning_set))

nonzero candidates: 2334


In [74]:
Mflat = matrix(QQ, [s.list() for s in spanning_set])
piv = Mflat.transpose().pivots()
basisT = [spanning_set[i] for i in piv]
print("dim T(G) =", len(basisT))

dim T(G) = 155


In [75]:
gens = Amats + Estar

def commutator_gram(basis, gens):
    D = []
    for Bmat in basis:
        row = []
        for g in gens:
            comm = Bmat*g - g*Bmat
            row += comm.list()
        D.append(row)
    Dmat = matrix(QQ, D)
    return Dmat * Dmat.transpose()

Gram = commutator_gram(basisT, gens)
center_coords = Gram.right_kernel().basis()
print("dim Z(T(G)) =", len(center_coords))

centerT = [sum(c[i]*basisT[i] for i in range(len(basisT))) for c in center_coords]

dim Z(T(G)) = 8


In [76]:
def express(X, basis_list):
    """Coordinates of X in the given basis."""
    A = matrix(QQ, [bb.list() for bb in basis_list]).transpose()
    return A.solve_right(vector(QQ, X.list()))

s = len(centerT)
Lmats = []
for i in range(s):
    cols = [express(centerT[i]*centerT[j], centerT) for j in range(s)]
    Lmats.append(matrix(QQ, cols).transpose())

In [77]:
Pdiag_Z = simultaneous_diagonalizer(Lmats)
Meig_Z  = eigenvalue_matrix(Pdiag_Z, Lmats)
epsilons = idempotents_from(Meig_Z, centerT)

print("all idempotent:  ", all((e*e - e).is_zero() for e in epsilons))
print("mutually orthogonal:", all((epsilons[i]*epsilons[j]).is_zero()
                                   for i in range(s) for j in range(s) if i != j))
print("sum == identity:  ", sum(epsilons) == identity_matrix(QQ, N))

all idempotent:   True
mutually orthogonal: True
sum == identity:   True


In [78]:
def block_dim(eps, basis):
    prods = matrix(QQ, [(Bmat*eps).list() for Bmat in basis])
    return prods.rank()

degrees_sq = [block_dim(e, basisT) for e in epsilons]
degrees = sorted(sqrt(dd) for dd in degrees_sq)

print("degrees:", degrees)
print("dim T(G) through idempotents:", sum(degrees_sq), " (should equal", len(basisT), ")")

degrees: [1, 1, 3, 3, 5, 5, 6, 7]
dim T(G) through idempotents: 155  (should equal 155 )


In [79]:
degrees_sq

[1, 1, 9, 25, 9, 36, 25, 49]